# Nail disease: ConvNeXt-Tiny - hyperparameter sweep (frozen / partial / full)

**Before running** (right-hand panel -> *Settings*):
1. **Accelerator:** GPU T4 (T4 x2 also works; only one GPU is used).
2. **Internet:** On (the pretrained weights are downloaded once).
3. **Add Input:** your private Kaggle dataset `nail_dataset` (the folder with `dataset_v2_main/`). The scripts find it automatically.

**Then click Save Version -> Save & Run All (Commit)**, check the accelerator in the dialog, and you can close the tab.
Expected time on a T4: about 1.5 h (estimate).

**When it finishes:** open the saved version -> **Output** tab -> download `nail_sweep_convnext_tiny.zip` (small) and send it.
Do **not** delete the version: it also holds the model checkpoints needed later for the explainability (XAI) work.

**What this notebook does**
1. *Tune (fold 0):* for each strategy it runs the default setting plus 3 trials, the same budget for every model.
2. *Extra experiments (optional):* your own `train.py` runs, see the cell below.
3. *Confirm (folds 1-4):* the best setting per strategy is re-run on the other four folds. Report these folds 1-4 numbers: those test images were never used to choose the setting.
4. *Report + zip.*

**Protocol:** there is no validation set, so every run trains for a fixed number of epochs and the **last epoch** is reported. Winners are picked on fold 0 by macro-F1, and a trial replaces the default only if it is at least 0.01 better.

**Changing hyperparameters later**
- Default settings of a model: `PRESETS` in the `common.py` cell.
- The 3 sweep trials of each strategy: `trials_for()` in the `sweep.py` cell.
- One-off experiments: the *Extra experiments* cell (no code changes needed).

In [ ]:
# ==================== SETTINGS: the only line that differs between the six notebooks ====================
MODEL = "convnext_tiny"     # resnet50 | densenet121 | vgg16 | convnext_tiny | vit_b16 | dinov2_b14
# ========================================================================================================

import subprocess, sys, os

def run(cmd):
    """Run a shell command, stream its output, and RAISE if it fails (used for the quick smoke test)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {cmd}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision, timm, sklearn
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| sklearn", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator to GPU T4 and restart the notebook"

## 1. Write the scripts
These cells create the five `.py` files in `/kaggle/working`. They are identical in all six notebooks.

In [ ]:
%%writefile common.py
"""Shared code for the nail-disease transfer-learning study.

Everything that must be identical across models and strategies lives here:
data loading, pad-to-square, augmentation, model construction, which layers
are trainable under each strategy, optimiser parameter groups, the LR
schedule and the metrics.
"""
import glob
import math
import os
import random

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.utils.data import Dataset
from torchvision.transforms import InterpolationMode
from torchvision.transforms import v2 as T

# --------------------------------------------------------------------------
# Model registry: all six models of the study.
#   timm_name:        pretrained weights to load through timm
#   partial_prefixes: parameter-name prefixes unfrozen by the "partial"
#                     strategy (the classifier head is always trainable and found automatically)
#   create_kwargs:    extra arguments for timm.create_model (optional)
# To add a model later: add an entry here and three lines in PRESETS below.
# --------------------------------------------------------------------------
MODEL_REGISTRY = {
    "resnet50": {
        "timm_name": "resnet50.tv_in1k",
        "partial_prefixes": ["layer4."],
    },
    "vgg16": {
        "timm_name": "vgg16.tv_in1k",
        # last conv block (block 5) = features.24 / 26 / 28
        "partial_prefixes": ["features.24.", "features.26.", "features.28."],
    },
    "densenet121": {
        "timm_name": "densenet121.tv_in1k",
        # last dense block + the final BatchNorm that follows it
        "partial_prefixes": ["features.denseblock4.", "features.norm5."],
    },
    "convnext_tiny": {
        "timm_name": "convnext_tiny.fb_in1k",
        # last stage (stage 3) + the LayerNorm in the head before the classifier
        "partial_prefixes": ["stages.3.", "head.norm."],
    },
    "vit_b16": {
        "timm_name": "vit_base_patch16_224.augreg_in1k",
        # last 4 of the 12 transformer blocks + the final LayerNorm
        "partial_prefixes": ["blocks.8.", "blocks.9.", "blocks.10.", "blocks.11.", "norm."],
    },
    "dinov2_b14": {
        "timm_name": "vit_base_patch14_dinov2.lvd142m",
        # same rule as ViT-B/16: last 4 of the 12 transformer blocks + the final LayerNorm
        "partial_prefixes": ["blocks.8.", "blocks.9.", "blocks.10.", "blocks.11.", "norm."],
        # timm's DINOv2 defaults to 518 px; img_size=224 resamples the position embeddings
        "create_kwargs": {"img_size": 224},
    },
}

# Settings shared by every run unless overridden on the command line.
COMMON_DEFAULTS = {
    "img_size": 224,
    "batch_size": 32,
    "loss": "weighted_ce",      # weighted_ce | ce | sampler
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "weight_decay": 0.01,
    "rrc_scale_min": 0.7,       # RandomResizedCrop keeps >= 70% of the image
    "rotation": 15.0,           # degrees
    "jitter": 0.1,              # brightness/contrast; saturation = jitter/2; hue = 0
    "drop_rate": 0.0,           # dropout before the classifier
    "min_lr_ratio": 0.01,       # cosine decays to 1% of the peak LR
    "seed": 42,
    "num_workers": 2,
}

# Starting hyperparameters per (model, strategy). Change these, or override
# any of them on the command line, to run hyperparameter experiments.
PRESETS = {
    ("resnet50", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("resnet50", "partial"): {"epochs": 40, "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup_epochs": 5},
    ("resnet50", "full"):    {"epochs": 50, "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup_epochs": 5},
    ("vgg16", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("vgg16", "partial"): {"epochs": 40, "lr_backbone": 2e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("vgg16", "full"):    {"epochs": 50, "lr_backbone": 1e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("densenet121", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("densenet121", "partial"): {"epochs": 40, "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup_epochs": 5},
    ("densenet121", "full"):    {"epochs": 50, "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup_epochs": 5},
    ("convnext_tiny", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("convnext_tiny", "partial"): {"epochs": 40, "lr_backbone": 5e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("convnext_tiny", "full"):    {"epochs": 50, "lr_backbone": 2e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("vit_b16", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("vit_b16", "partial"): {"epochs": 40, "lr_backbone": 3e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("vit_b16", "full"):    {"epochs": 50, "lr_backbone": 1e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("dinov2_b14", "frozen"):  {"epochs": 30, "lr_backbone": 0.0,  "lr_head": 1e-3, "warmup_epochs": 3},
    ("dinov2_b14", "partial"): {"epochs": 40, "lr_backbone": 2e-5, "lr_head": 1e-3, "warmup_epochs": 5},
    ("dinov2_b14", "full"):    {"epochs": 50, "lr_backbone": 1e-5, "lr_head": 1e-3, "warmup_epochs": 5},
}


# --------------------------------------------------------------------------
# Paths and data
# --------------------------------------------------------------------------
def _is_dataset_dir(d):
    return (os.path.isfile(os.path.join(d, "manifest.csv"))
            and os.path.isfile(os.path.join(d, "folds.csv"))
            and os.path.isdir(os.path.join(d, "images")))


def _walk_for_dataset(root, max_depth=6):
    """Breadth-limited search under root for a folder holding manifest.csv + folds.csv + images/."""
    root = os.path.abspath(root)
    base_depth = root.rstrip(os.sep).count(os.sep)
    found = []
    for cur, dirs, _files in os.walk(root, followlinks=True):
        if _is_dataset_dir(cur):
            found.append(cur)
            dirs[:] = []                      # do not descend into images/ etc.
            continue
        if cur.count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
    return found


def find_data_dir(hint=None):
    """Locate dataset_v2_main (the folder with manifest.csv, folds.csv and images/).

    Order: --data_dir argument -> $NAIL_DATA_DIR -> Kaggle inputs (nail_dataset first)
    -> current directory and its parents. Works for /kaggle/input/nail_dataset/,
    /kaggle/input/nail-dataset/dataset_v2_main/, nested copies, and local folders.
    """
    candidates = [hint, os.environ.get("NAIL_DATA_DIR")]
    for c in candidates:
        if c:
            if _is_dataset_dir(c):
                return c
            hits = _walk_for_dataset(c) if os.path.isdir(c) else []
            if hits:
                return sorted(hits, key=len)[0]
            raise FileNotFoundError(
                f"No dataset (manifest.csv + folds.csv + images/) found at or under {c}")

    roots = []
    if os.path.isdir("/kaggle/input"):
        subs = sorted(os.listdir("/kaggle/input"))
        # put folders that look like our dataset first
        subs.sort(key=lambda n: (0 if "nail" in n.lower() else 1, n))
        roots += [os.path.join("/kaggle/input", n) for n in subs]
    roots += [".", "..", "nail_dataset", "../nail_dataset"]

    for root in roots:
        if os.path.isdir(root):
            hits = _walk_for_dataset(root)
            if hits:
                hits.sort(key=lambda p: (0 if os.path.basename(p) == "dataset_v2_main" else 1, len(p)))
                if len(hits) > 1:
                    print(f"NOTE: several dataset folders found, using {hits[0]} (others: {hits[1:]})")
                return hits[0]

    listing = ""
    if os.path.isdir("/kaggle/input"):
        listing = "\n/kaggle/input contains: " + ", ".join(sorted(os.listdir("/kaggle/input")) or ["(empty)"])
    raise FileNotFoundError(
        "Could not find dataset_v2_main (needs manifest.csv, folds.csv and an images/ folder)."
        + listing + "\nAdd the dataset via 'Add Input' in the notebook, or pass --data_dir PATH.")


def slug(name):
    """'Darier's disease' -> 'darier_s_disease' (safe for column names)."""
    return "".join(c if c.isalnum() else "_" for c in name.lower()).strip("_")


def class_list(df):
    """Fixed alphabetical class order, shared by every script."""
    return sorted(df["class"].unique())


def pad_to_square(img, fill):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new("RGB", (s, s), fill)
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas


def mean_fill(data_cfg):
    """Pad colour = the model's normalisation mean, so padding becomes ~0 after normalising."""
    return tuple(int(round(m * 255)) for m in data_cfg["mean"])


def build_transforms(data_cfg, cfg, train):
    size = cfg["img_size"]
    bicubic = InterpolationMode.BICUBIC
    if train:
        t = [
            T.RandomResizedCrop(size, scale=(cfg["rrc_scale_min"], 1.0), ratio=(0.9, 1.1),
                                interpolation=bicubic, antialias=True),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(cfg["rotation"], interpolation=InterpolationMode.BILINEAR,
                             fill=mean_fill(data_cfg)),
        ]
        if cfg["jitter"] > 0:
            j = cfg["jitter"]
            t.append(T.ColorJitter(brightness=j, contrast=j, saturation=j / 2, hue=0.0))
    else:
        t = [T.Resize((size, size), interpolation=bicubic, antialias=True)]
    t += [T.ToImage(), T.ToDtype(torch.float32, scale=True),
          T.Normalize(data_cfg["mean"], data_cfg["std"])]
    return T.Compose(t)


class NailDataset(Dataset):
    """Preloads every image once (padded to square) so no disk I/O happens per epoch."""

    def __init__(self, df, data_dir, class_to_idx, transform, fill):
        self.df = df.reset_index(drop=True)
        self.labels = [class_to_idx[c] for c in self.df["class"]]
        self.images = [pad_to_square(Image.open(os.path.join(data_dir, p)).convert("RGB"), fill)
                       for p in self.df["path"]]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]


# --------------------------------------------------------------------------
# Models and strategies
# --------------------------------------------------------------------------
def build_model(model_key, num_classes, pretrained=True, drop_rate=0.0):
    spec = MODEL_REGISTRY[model_key]
    model = timm.create_model(spec["timm_name"], pretrained=pretrained,
                              num_classes=num_classes, drop_rate=drop_rate,
                              **spec.get("create_kwargs", {}))
    data_cfg = timm.data.resolve_model_data_config(model)
    return model, data_cfg


def head_param_names(model):
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    return {n for n, p in model.named_parameters() if id(p) in head_ids}


def apply_strategy(model, model_key, strategy):
    """Set requires_grad according to the strategy. Returns the head parameter names."""
    head = head_param_names(model)
    prefixes = MODEL_REGISTRY[model_key]["partial_prefixes"]
    for n, p in model.named_parameters():
        if strategy == "full":
            p.requires_grad = True
        elif strategy == "partial":
            p.requires_grad = n in head or any(n.startswith(pr) for pr in prefixes)
        elif strategy == "frozen":
            p.requires_grad = n in head
        else:
            raise ValueError(strategy)
    return head


def set_train_mode(model, strategy):
    """model.train(), but BatchNorm layers whose weights are frozen stay in eval mode,
    so their ImageNet running statistics are not overwritten by our tiny batches."""
    model.train()
    if strategy != "full":
        for m in model.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                own = list(m.parameters(recurse=False))
                if own and not any(p.requires_grad for p in own):
                    m.eval()


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def make_param_groups(model, head_names, lr_backbone, lr_head, weight_decay):
    """Head and backbone get separate LRs; no weight decay on biases and norm weights."""
    buckets = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_head = n in head_names
        no_decay = p.ndim <= 1 or n.endswith(".bias")
        buckets.setdefault((is_head, no_decay), []).append(p)
    groups = []
    for (is_head, no_decay), params in buckets.items():
        groups.append({
            "params": params,
            "lr": lr_head if is_head else lr_backbone,
            "weight_decay": 0.0 if no_decay else weight_decay,
            "group_name": ("head" if is_head else "backbone") + ("_nodecay" if no_decay else ""),
        })
    return groups


def warmup_cosine(optimizer, total_steps, warmup_steps, min_ratio=0.01):
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    w = 1.0 / np.maximum(counts, 1)
    return w / w.mean()


# --------------------------------------------------------------------------
# Misc
# --------------------------------------------------------------------------
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


def compute_metrics(y_true, y_pred, num_classes):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro",
                             labels=list(range(num_classes)), zero_division=0),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
    }


def load_split(split_csv):
    df = pd.read_csv(split_csv)
    need = {"id", "path", "class", "source_dataset", "split", "test_fold"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{split_csv} is missing columns {missing}; run split_dataset.py first")
    return df


In [ ]:
%%writefile split_dataset.py
"""Check the dataset and write a grouped train/test split.

The split reuses the dataset's own StratifiedGroupKFold folds: images whose
fold == --test_fold become the test set (~20%), all others the training set.
This keeps near-duplicates and related images on one side only, which a fresh
random split would not.

Usage:
    python split_dataset.py --test_fold 0
"""
import argparse
import json
import os
import sys

import numpy as np
import pandas as pd

from common import class_list, find_data_dir


def fail(msg):
    print(f"CHECK FAILED: {msg}")
    sys.exit(1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default=None)
    ap.add_argument("--test_fold", type=int, default=0)
    ap.add_argument("--out_dir", default="splits")
    args = ap.parse_args()

    data_dir = find_data_dir(args.data_dir)
    print(f"Dataset folder: {data_dir}")
    m = pd.read_csv(os.path.join(data_dir, "manifest.csv"))
    f = pd.read_csv(os.path.join(data_dir, "folds.csv"))

    # ---- integrity checks -------------------------------------------------
    if len(m) != 1008:
        print(f"WARNING: expected 1008 images, manifest has {len(m)}")
    if set(m["id"]) != set(f["id"]) or m["id"].duplicated().any():
        fail("manifest.csv and folds.csv do not contain the same unique ids")
    j = m.merge(f, on="id", suffixes=("", "_folds"), validate="one_to_one")
    for col in ["fold", "path", "class", "source_dataset", "group_id"]:
        if col + "_folds" in j and not (j[col] == j[col + "_folds"]).all():
            bad = j.loc[j[col] != j[col + "_folds"], ["id", col, col + "_folds"]]
            fail(f"'{col}' differs between manifest and folds.csv:\n{bad.head()}")
    print("OK  folds.csv agrees with the manifest (fold, path, class, source, group)")

    spans = m.groupby("group_id")["fold"].nunique()
    if (spans > 1).any():
        fail(f"{(spans > 1).sum()} groups span more than one fold")
    print(f"OK  no group spans folds ({m['group_id'].nunique()} groups)")

    missing = [p for p in m["path"] if not os.path.exists(os.path.join(data_dir, p))]
    if missing:
        fail(f"{len(missing)} image files missing, e.g. {missing[:3]}")
    print("OK  all image files exist")

    if args.test_fold not in set(m["fold"]):
        fail(f"test_fold {args.test_fold} not in folds {sorted(m['fold'].unique())}")

    # ---- write split ------------------------------------------------------
    cols = ["id", "path", "class", "source_dataset", "group_id", "fold",
            "orig_width", "orig_height", "confidence", "review_flag"]
    out = m[cols].copy()
    out["split"] = np.where(out["fold"] == args.test_fold, "test", "train")
    out["test_fold"] = args.test_fold

    os.makedirs(args.out_dir, exist_ok=True)
    split_path = os.path.join(args.out_dir, f"split_fold{args.test_fold}.csv")
    out.to_csv(split_path, index=False)
    classes = class_list(out)
    with open(os.path.join(args.out_dir, "class_map.json"), "w") as fh:
        json.dump({c: i for i, c in enumerate(classes)}, fh, indent=2)

    n_tr, n_te = (out["split"] == "train").sum(), (out["split"] == "test").sum()
    print(f"\nWrote {split_path}: train {n_tr} / test {n_te} ({n_te / len(out):.1%} test)")
    print("\nImages per class:")
    print(pd.crosstab(out["class"], out["split"]).to_string())
    print("\nImages per source:")
    print(pd.crosstab(out["source_dataset"], out["split"]).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Train one (model, strategy) on the train split and evaluate on the test split.

There is no validation set, so the model is trained for a FIXED number of
epochs and the LAST epoch is the one reported and saved. Test metrics are
logged every epoch only to draw learning curves; never pick the epoch from them.

Examples:
    python train.py --model resnet50 --strategy frozen
    python train.py --model resnet50 --strategy partial
    python train.py --model resnet50 --strategy full
    # hyperparameter experiment, saved separately under its own tag:
    python train.py --model resnet50 --strategy full --lr_backbone 3e-5 --tag lr3e-5
"""
import argparse
import json
import os
import time

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler

from common import (COMMON_DEFAULTS, MODEL_REGISTRY, PRESETS, NailDataset, apply_strategy,
                    build_model, build_transforms, class_list, class_weights, compute_metrics,
                    count_params, find_data_dir, load_split, make_param_groups, mean_fill,
                    seed_everything, seed_worker, set_train_mode, slug, warmup_cosine)

OVERRIDABLE = {
    "epochs": int, "lr_backbone": float, "lr_head": float, "weight_decay": float,
    "warmup_epochs": int, "batch_size": int, "label_smoothing": float, "grad_clip": float,
    "rrc_scale_min": float, "rotation": float, "jitter": float, "drop_rate": float,
    "min_lr_ratio": float, "seed": int, "num_workers": int,
}


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="resnet50", choices=sorted(MODEL_REGISTRY))
    ap.add_argument("--strategy", required=True, choices=["frozen", "partial", "full"])
    ap.add_argument("--split_csv", default="splits/split_fold0.csv")
    ap.add_argument("--data_dir", default=None)
    ap.add_argument("--out_dir", default="runs")
    ap.add_argument("--tag", default="default", help="name for this hyperparameter setting")
    for k, typ in OVERRIDABLE.items():
        ap.add_argument(f"--{k}", type=typ, default=None)
    ap.add_argument("--loss", choices=["weighted_ce", "ce", "sampler"], default=None)
    ap.add_argument("--save_checkpoint", choices=["full", "trainable", "none"], default="full")
    ap.add_argument("--no_pretrained", action="store_true", help="debug only")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--overwrite", action="store_true", help="rerun even if DONE exists")
    return ap.parse_args()


def build_config(args):
    cfg = dict(COMMON_DEFAULTS)
    cfg.update(PRESETS[(args.model, args.strategy)])
    for k in list(OVERRIDABLE) + ["loss"]:
        v = getattr(args, k)
        if v is not None:
            cfg[k] = v
    cfg.update(model=args.model, strategy=args.strategy, tag=args.tag,
               timm_name=MODEL_REGISTRY[args.model]["timm_name"],
               pretrained=not args.no_pretrained)
    if args.strategy == "frozen":
        cfg["lr_backbone"] = 0.0
    return cfg


@torch.no_grad()
def predict(model, loader, device, use_amp):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        probs.append(F.softmax(logits.float(), dim=1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


@torch.no_grad()
def inference_latency_ms(model, device, size, n=50):
    """Median single-image latency in fp32 (same measurement for every model)."""
    model.eval()
    x = torch.randn(1, 3, size, size, device=device).to(memory_format=torch.channels_last)
    for _ in range(10):
        model(x)
    times = []
    for _ in range(n):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def main():
    args = parse_args()
    cfg = build_config(args)
    split = load_split(args.split_csv)
    fold = int(split["test_fold"].iloc[0])
    run_dir = os.path.join(args.out_dir, args.model, args.strategy, args.tag, f"fold{fold}")
    if os.path.exists(os.path.join(run_dir, "DONE")) and not args.overwrite:
        print(f"SKIP {run_dir} (already done; use --overwrite to rerun)")
        return
    os.makedirs(run_dir, exist_ok=True)

    seed_everything(cfg["seed"])
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_amp

    data_dir = find_data_dir(args.data_dir)
    classes = class_list(split)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    K = len(classes)
    train_df = split[split["split"] == "train"]
    test_df = split[split["split"] == "test"]

    # ---- model + strategy --------------------------------------------------
    model, data_cfg = build_model(args.model, K, pretrained=cfg["pretrained"],
                                  drop_rate=cfg["drop_rate"])
    head_names = apply_strategy(model, args.model, args.strategy)
    model = model.to(device).to(memory_format=torch.channels_last)
    n_total, n_trainable = count_params(model)

    # ---- data --------------------------------------------------------------
    fill = mean_fill(data_cfg)
    train_ds = NailDataset(train_df, data_dir, class_to_idx,
                           build_transforms(data_cfg, cfg, train=True), fill)
    test_ds = NailDataset(test_df, data_dir, class_to_idx,
                          build_transforms(data_cfg, cfg, train=False), fill)
    g = torch.Generator().manual_seed(cfg["seed"])
    weights = class_weights(np.array(train_ds.labels), K)
    if cfg["loss"] == "sampler":
        sample_w = weights[np.array(train_ds.labels)]
        sampler = WeightedRandomSampler(sample_w, num_samples=len(train_ds),
                                        replacement=True, generator=g)
        shuffle = False
    else:
        sampler, shuffle = None, True
    loader_kw = dict(num_workers=cfg["num_workers"], pin_memory=device.type == "cuda",
                     persistent_workers=cfg["num_workers"] > 0, worker_init_fn=seed_worker)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=shuffle,
                              sampler=sampler, drop_last=True, generator=g, **loader_kw)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, **loader_kw)

    loss_w = torch.tensor(weights, dtype=torch.float32, device=device) \
        if cfg["loss"] == "weighted_ce" else None
    criterion = nn.CrossEntropyLoss(weight=loss_w, label_smoothing=cfg["label_smoothing"])

    # ---- optimiser + schedule ----------------------------------------------
    groups = make_param_groups(model, head_names, cfg["lr_backbone"], cfg["lr_head"],
                               cfg["weight_decay"])
    optimizer = torch.optim.AdamW(groups)
    steps_per_epoch = len(train_loader)
    scheduler = warmup_cosine(optimizer, cfg["epochs"] * steps_per_epoch,
                              cfg["warmup_epochs"] * steps_per_epoch, cfg["min_lr_ratio"])
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    trainable = [p for p in model.parameters() if p.requires_grad]

    print(f"Run: {run_dir}")
    print(f"Device {device} | AMP {use_amp} | train {len(train_ds)} / test {len(test_ds)} | "
          f"params {n_total / 1e6:.2f}M total, {n_trainable / 1e6:.3f}M trainable")
    print("Config:", json.dumps({k: v for k, v in cfg.items() if k != 'timm_name'}))

    # ---- training ----------------------------------------------------------
    log = []
    t_start = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()
        set_train_mode(model, args.strategy)
        tot_loss, tot_correct, tot_n = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg["grad_clip"] > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tot_loss += loss.item() * len(y)
            tot_correct += (logits.argmax(1) == y).sum().item()
            tot_n += len(y)

        probs, y_test = predict(model, test_loader, device, use_amp)
        test_loss = F.nll_loss(torch.log(torch.from_numpy(probs).clamp_min(1e-8)),
                               torch.from_numpy(y_test)).item()
        mt = compute_metrics(y_test, probs.argmax(1), K)
        lrs = {grp["group_name"]: grp["lr"] for grp in optimizer.param_groups}
        row = {"epoch": epoch, "train_loss": tot_loss / tot_n, "train_acc": tot_correct / tot_n,
               "test_loss": test_loss, "test_acc": mt["acc"], "test_macro_f1": mt["macro_f1"],
               "test_bal_acc": mt["bal_acc"],
               "lr_head": lrs.get("head", 0.0), "lr_backbone": lrs.get("backbone", 0.0),
               "epoch_time_s": time.time() - t0}
        log.append(row)
        pd.DataFrame(log).to_csv(os.path.join(run_dir, "epoch_log.csv"), index=False)
        print(f"ep {epoch:3d}/{cfg['epochs']} | train loss {row['train_loss']:.3f} "
              f"acc {row['train_acc']:.3f} | test acc {mt['acc']:.3f} "
              f"macroF1 {mt['macro_f1']:.3f} balAcc {mt['bal_acc']:.3f} | "
              f"{row['epoch_time_s']:.1f}s")
    train_time = time.time() - t_start

    # ---- final (last-epoch) predictions ------------------------------------
    pred = probs.argmax(1)
    out = pd.DataFrame({
        "image_id": test_df["id"].values, "fold": fold, "model": args.model,
        "strategy": args.strategy, "tag": args.tag, "seed": cfg["seed"],
        "source_dataset": test_df["source_dataset"].values,
        "true_idx": y_test, "true_class": [classes[i] for i in y_test],
        "pred_idx": pred, "pred_class": [classes[i] for i in pred],
    })
    for k, c in enumerate(classes):
        out[f"prob_{slug(c)}"] = probs[:, k].round(6)
    out.to_csv(os.path.join(run_dir, "predictions.csv"), index=False)

    # ---- checkpoint ----------------------------------------------------------
    ckpt_mb = 0.0
    if args.save_checkpoint != "none":
        state = model.state_dict()
        if args.save_checkpoint == "trainable":
            keep = {n for n, p in model.named_parameters() if p.requires_grad}
            state = {k: v for k, v in state.items() if k in keep}
        ckpt_path = os.path.join(run_dir, "model.pt")
        torch.save({"state_dict": state, "classes": classes, "config": cfg,
                    "timm_name": cfg["timm_name"], "saved": args.save_checkpoint}, ckpt_path)
        ckpt_mb = os.path.getsize(ckpt_path) / 1e6

    final = compute_metrics(y_test, pred, K)
    summary = {
        "model": args.model, "strategy": args.strategy, "tag": args.tag, "fold": fold,
        "final_epoch": cfg["epochs"], **{f"test_{k}": v for k, v in final.items()},
        "train_time_min": train_time / 60, "params_total": n_total,
        "params_trainable": n_trainable, "checkpoint_mb": ckpt_mb,
        "latency_ms_bs1_fp32": inference_latency_ms(model, device, cfg["img_size"]),
        "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
        "torch": torch.__version__, "timm": timm.__version__,
        "pretrained_license": model.pretrained_cfg.get("license"),
    }
    with open(os.path.join(run_dir, "config.json"), "w") as fh:
        json.dump(cfg, fh, indent=2)
    with open(os.path.join(run_dir, "summary.json"), "w") as fh:
        json.dump(summary, fh, indent=2)
    open(os.path.join(run_dir, "DONE"), "w").close()
    print(f"\nFINAL (epoch {cfg['epochs']}): acc {final['acc']:.3f} | "
          f"macro-F1 {final['macro_f1']:.3f} | bal-acc {final['bal_acc']:.3f} | "
          f"train time {train_time / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Evaluate every finished run under runs/ and compare strategies.

Writes to results/:
  summary.csv            one row per run (+ metadata-only baseline row)
  per_class_recall.csv   recall of each class for each run
  source_breakdown.csv   BASE vs external, and per-source recall for healthy/clubbing/psoriasis
  mcnemar.csv            exact McNemar test between every pair of runs on the same fold
  predictions_all.csv    all per-image predictions combined (the file to send for evaluation)
  confusion_<run>.png    confusion matrix per run
  learning_curves.png    train loss and test macro-F1 per epoch

Usage:
    python evaluate.py
"""
import argparse
import glob
import itertools
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, recall_score

from common import class_list, compute_metrics, find_data_dir, load_split

FOCUS = ["healthy", "clubbing", "psoriasis"]  # classes that mix BASE and external sources


def run_name(s):
    return f"{s['model']}-{s['strategy']}-{s['tag']}-f{s['fold']}"


def metadata_baseline(split, data_dir, classes):
    """Random forest on PNG width, height, aspect ratio and area only (no pixels)."""
    feats = []
    for p in split["path"]:
        w, h = Image.open(os.path.join(data_dir, p)).size
        feats.append([w, h, w / h, w * h])
    X = np.array(feats, dtype=float)
    y = split["class"].map({c: i for i, c in enumerate(classes)}).values
    tr, te = (split["split"] == "train").values, (split["split"] == "test").values
    rf = RandomForestClassifier(n_estimators=500, random_state=0, class_weight="balanced")
    rf.fit(X[tr], y[tr])
    return compute_metrics(y[te], rf.predict(X[te]), len(classes))


def plot_confusion(cm, classes, title, path):
    cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)), classes, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(classes)), classes, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                        color="white" if cm_norm[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", default="runs")
    ap.add_argument("--split_csv", default="splits/split_fold0.csv")
    ap.add_argument("--data_dir", default=None)
    ap.add_argument("--out_dir", default="results")
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    split = load_split(args.split_csv)
    classes = class_list(split)
    K = len(classes)
    data_dir = find_data_dir(args.data_dir)

    run_dirs = sorted(os.path.dirname(p) for p in
                      glob.glob(os.path.join(args.runs_dir, "*", "*", "*", "fold*", "DONE")))
    if not run_dirs:
        raise SystemExit(f"No finished runs found under {args.runs_dir}")

    rows, recall_rows, source_rows, preds_all, curves = [], [], [], [], {}
    preds_by_run = {}
    for d in run_dirs:
        s = json.load(open(os.path.join(d, "summary.json")))
        pr = pd.read_csv(os.path.join(d, "predictions.csv"))
        name = run_name(s)
        y, p = pr["true_idx"].values, pr["pred_idx"].values
        preds_by_run[name] = (s["fold"], pr.set_index("image_id")["true_idx"],
                              pr.set_index("image_id")["pred_idx"])
        preds_all.append(pr)
        curves[name] = pd.read_csv(os.path.join(d, "epoch_log.csv"))

        m = compute_metrics(y, p, K)
        is_base = pr["source_dataset"].eq("BASE").values
        rows.append({
            "run": name, "model": s["model"], "strategy": s["strategy"], "tag": s["tag"],
            "fold": s["fold"], "n_test": len(y),
            "acc": m["acc"], "macro_f1": m["macro_f1"], "bal_acc": m["bal_acc"],
            "acc_BASE": (y[is_base] == p[is_base]).mean(),
            "acc_external": (y[~is_base] == p[~is_base]).mean(),
            "params_trainable_M": s["params_trainable"] / 1e6,
            "params_total_M": s["params_total"] / 1e6,
            "train_time_min": s["train_time_min"], "latency_ms": s["latency_ms_bs1_fp32"],
            "checkpoint_mb": s["checkpoint_mb"], "epochs": s["final_epoch"],
        })

        rec = recall_score(y, p, labels=list(range(K)), average=None, zero_division=0)
        recall_rows.append({"run": name, **{c: r for c, r in zip(classes, rec)}})

        for cls in FOCUS:
            for src, sub in pr[pr["true_class"] == cls].groupby("source_dataset"):
                source_rows.append({"run": name, "class": cls, "source": src, "n": len(sub),
                                    "recall": (sub["pred_class"] == cls).mean()})
        for grp, mask in [("BASE", is_base), ("external", ~is_base)]:
            source_rows.append({"run": name, "class": "ALL (accuracy)", "source": grp,
                                "n": int(mask.sum()), "recall": (y[mask] == p[mask]).mean()})

        cm = confusion_matrix(y, p, labels=list(range(K)))
        plot_confusion(cm, classes, f"{name}  (acc {m['acc']:.2f}, macro-F1 {m['macro_f1']:.2f})",
                       os.path.join(args.out_dir, f"confusion_{name}.png"))

    # metadata-only baseline (same split)
    mb = metadata_baseline(split, data_dir, classes)
    rows.append({"run": "metadata_only_baseline", "model": "random_forest",
                 "strategy": "PNG size + aspect", "tag": "-", "fold": int(split["test_fold"].iloc[0]),
                 "n_test": int((split["split"] == "test").sum()),
                 "acc": mb["acc"], "macro_f1": mb["macro_f1"], "bal_acc": mb["bal_acc"]})
    majority = split.loc[split["split"] == "test", "class"].value_counts(normalize=True).iloc[0]

    order = {"frozen": 0, "partial": 1, "full": 2}
    summary = pd.DataFrame(rows)
    summary["_o"] = [(0 if r != "metadata_only_baseline" else 1, m, order.get(st, 9), t)
                     for r, m, st, t in zip(summary["run"], summary["model"],
                                            summary["strategy"], summary["tag"])]
    summary = summary.sort_values("_o").drop(columns="_o")
    summary.to_csv(os.path.join(args.out_dir, "summary.csv"), index=False)
    pd.DataFrame(recall_rows).to_csv(os.path.join(args.out_dir, "per_class_recall.csv"), index=False)
    pd.DataFrame(source_rows).to_csv(os.path.join(args.out_dir, "source_breakdown.csv"), index=False)
    pd.concat(preds_all).to_csv(os.path.join(args.out_dir, "predictions_all.csv"), index=False)

    # exact McNemar between runs on the same fold
    mc = []
    for a, b in itertools.combinations(preds_by_run, 2):
        fa, ya, pa = preds_by_run[a]
        fb, yb, pb = preds_by_run[b]
        if fa != fb:
            continue
        ids = ya.index.intersection(yb.index)
        ca = (pa[ids] == ya[ids]).values
        cb = (pb[ids] == yb[ids]).values
        n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
        pval = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        mc.append({"run_a": a, "run_b": b, "a_right_b_wrong": n01, "a_wrong_b_right": n10,
                   "p_value_exact": pval})
    pd.DataFrame(mc).to_csv(os.path.join(args.out_dir, "mcnemar.csv"), index=False)

    # learning curves
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, c in curves.items():
        axes[0].plot(c["epoch"], c["train_loss"], label=name)
        axes[1].plot(c["epoch"], c["test_macro_f1"], label=name)
    axes[0].set_title("Train loss")
    axes[1].set_title("Test macro-F1 per epoch (curves only; report the last epoch)")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out_dir, "learning_curves.png"), dpi=120)
    plt.close(fig)

    pd.set_option("display.width", 200)
    cols = ["run", "acc", "macro_f1", "bal_acc", "acc_BASE", "acc_external",
            "params_trainable_M", "train_time_min", "latency_ms"]
    print(summary[cols].round(3).to_string(index=False))
    print(f"\nMajority-class baseline accuracy on this test split: {majority:.3f}")
    if mc:
        print("\nMcNemar (exact):")
        print(pd.DataFrame(mc).round(4).to_string(index=False))
    print(f"\nWrote results to {args.out_dir}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sweep.py
"""Hyperparameter sweep runner for the nail-disease study.

Every configuration (model x strategy) gets the SAME tuning budget: the default
run plus 3 trials. Trials are scored on fold 0 (the test fold used so far), the
winner is then re-run on folds 1-4, and the mean over folds 1-4 is the number you
report: those test images were never used to choose the hyperparameters.

Phases (run them in order, one model per notebook):
    python sweep.py --model convnext_tiny --phase tune       # default + 3 trials x 3 strategies on fold 0
    python sweep.py --model convnext_tiny --phase confirm    # winners on folds 1-4
    python sweep.py --phase report --runs_dir runs           # table over all models found in runs_dir

Changing things later:
  * the default settings of a model        -> PRESETS in common.py
  * the 3 trials of each strategy          -> trials_for() below
  * one-off experiments                    -> run train.py with your own --tag on fold 0 BEFORE the
                                              confirm phase; if its fold-0 macro-F1 is the best it is
                                              picked up and confirmed on folds 1-4 automatically
  * repeating a run with another seed      -> use a tag starting with "seed_" (ignored by the selection)

Add --dry_run to print the commands without running anything.
A finished run (DONE file) is skipped, so an interrupted session can just be restarted.
"""
import argparse
import glob
import json
import os
import subprocess
import sys

import pandas as pd

def base_lrs(model):
    """LRs of the 'default' run, read from PRESETS in common.py so the two can never disagree.
    head = LR of the new classifier layer; partial / full = LR of the unfrozen backbone layers."""
    from common import MODEL_REGISTRY, PRESETS          # lazy: needs torch/timm, report phase does not
    if model not in MODEL_REGISTRY:
        raise SystemExit(f"Unknown model '{model}'. Known: {sorted(MODEL_REGISTRY)}")
    return {"head": PRESETS[(model, "frozen")]["lr_head"],
            "partial": PRESETS[(model, "partial")]["lr_backbone"],
            "full": PRESETS[(model, "full")]["lr_backbone"]}


STRATEGIES = ["frozen", "partial", "full"]
NOT_CANDIDATES = ("seed_",)   # tags starting with this are ignored by the winner selection
MIN_GAIN = 0.01          # a trial must beat the default by >= 1 point macro-F1 (~2 test images)
FOLDS_CONFIRM = [1, 2, 3, 4]

REG = ["--rrc_scale_min", "0.5", "--rotation", "25", "--jitter", "0.15",
       "--weight_decay", "0.05", "--drop_rate", "0.3"]
LIGHT_AUG = ["--rrc_scale_min", "0.9", "--rotation", "5", "--jitter", "0.05"]


def g(x):
    return f"{x:.3g}"


def trials_for(model, strategy):
    """[(tag, extra train.py args)] -> the default plus exactly 3 trials."""
    b = base_lrs(model)
    t = [("default", [])]
    if strategy == "frozen":
        # frozen heads are under-trained at the default (train acc ~0.8): higher LR, longer, lighter aug
        t += [("h_x3", ["--lr_head", g(b["head"] * 3)]),
              ("h_x3_e60", ["--lr_head", g(b["head"] * 3), "--epochs", "60"]),
              ("h_x3_e60_lightaug", ["--lr_head", g(b["head"] * 3), "--epochs", "60"] + LIGHT_AUG)]
    else:
        # partial / full reach ~100% train accuracy by epoch ~20: LR down/up, and a stronger-regularisation recipe
        lr = b[strategy]
        t += [("lr_x0.3", ["--lr_backbone", g(lr * 0.3)]),
              ("lr_x3", ["--lr_backbone", g(lr * 3)]),
              ("reg", REG)]
    return t


def run(cmd, dry):
    print("$", " ".join(cmd), flush=True)
    if dry:
        return 0
    return subprocess.call(cmd)


def ensure_split(fold, dry):
    path = f"splits/split_fold{fold}.csv"
    if not os.path.exists(path):
        run([sys.executable, "split_dataset.py", "--test_fold", str(fold)], dry)
    return path


def load_summaries(runs_dir):
    rows = []
    for p in glob.glob(os.path.join(runs_dir, "*", "*", "*", "fold*", "summary.json")):
        if not os.path.exists(os.path.join(os.path.dirname(p), "DONE")):
            continue
        s = json.load(open(p))
        rows.append({"model": s["model"], "strategy": s["strategy"], "tag": s["tag"], "fold": s["fold"],
                     "acc": s["test_acc"], "macro_f1": s["test_macro_f1"], "bal_acc": s["test_bal_acc"]})
    return pd.DataFrame(rows, columns=["model", "strategy", "tag", "fold", "acc", "macro_f1", "bal_acc"])


def select_tag(df, model, strategy):
    """Winner on fold 0: best macro-F1 (ties -> accuracy); keep 'default' unless beaten by >= MIN_GAIN."""
    d = df[(df.model == model) & (df.strategy == strategy) & (df.fold == 0)]
    d = d[~d.tag.str.startswith(NOT_CANDIDATES)]
    if d.empty:
        return None
    best = d.sort_values(["macro_f1", "acc"], ascending=False).iloc[0]
    base = d[d.tag == "default"]
    if len(base) and best.macro_f1 - base.iloc[0].macro_f1 < MIN_GAIN:
        return "default"
    return best.tag


def prune_losers(args, strategy, selected):
    """Delete model.pt of tuning trials that did not win (keeps 'default' and the winner) to save disk."""
    freed = 0
    for p in glob.glob(os.path.join(args.runs_dir, args.model, strategy, "*", "fold0", "model.pt")):
        tag = p.split(os.sep)[-3]
        if tag in ("default", selected):
            continue
        freed += os.path.getsize(p)
        if not args.dry_run:
            os.remove(p)
    if freed:
        print(f"  pruned {freed / 1e6:.0f} MB of checkpoints from losing trials ({strategy})")


def phase_tune(args):
    failed = []
    ensure_split(0, args.dry_run)
    for st in STRATEGIES:
        for tag, extra in trials_for(args.model, st):
            cmd = [sys.executable, "train.py", "--model", args.model, "--strategy", st, "--tag", tag,
                   "--split_csv", "splits/split_fold0.csv", "--out_dir", args.runs_dir,
                   "--save_checkpoint", "full"] + extra
            if run(cmd, args.dry_run) != 0:
                failed.append((st, tag))
    if failed:
        print("\nFAILED:", failed)
    print("\nNext: python evaluate.py   (compares all tags on fold 0), then --phase confirm")


CONFIG_KEYS = ["epochs", "lr_backbone", "lr_head", "weight_decay", "warmup_epochs", "batch_size",
               "label_smoothing", "grad_clip", "rrc_scale_min", "rotation", "jitter", "drop_rate",
               "min_lr_ratio", "seed", "num_workers", "loss"]


def args_from_config(runs_dir, model, strategy, tag):
    """Exact train.py arguments of a finished fold-0 run, read from its config.json."""
    path = os.path.join(runs_dir, model, strategy, tag, "fold0", "config.json")
    if not os.path.exists(path):
        return []
    cfg = json.load(open(path))
    out = []
    for k in CONFIG_KEYS:
        if k in cfg:
            out += [f"--{k}", str(cfg[k])]
    return out


def phase_confirm(args):
    failed = []
    df = load_summaries(args.runs_dir)
    for st in STRATEGIES:
        tag = select_tag(df, args.model, st)
        if tag is None:
            print(f"[{st}] no fold-0 results yet; run --phase tune first")
            continue
        extra = args_from_config(args.runs_dir, args.model, st, tag)
        tags = [(tag, extra)]
        if args.include_default and tag != "default":
            tags.append(("default", []))
        print(f"\n[{args.model} / {st}] selected tag on fold 0: {tag}")
        for t, ex in tags:
            for fold in FOLDS_CONFIRM:
                split = ensure_split(fold, args.dry_run)
                cmd = [sys.executable, "train.py", "--model", args.model, "--strategy", st, "--tag", t,
                       "--split_csv", split, "--out_dir", args.runs_dir, "--save_checkpoint", "full"] + ex
                if run(cmd, args.dry_run) != 0:
                    failed.append((st, t, fold))
        if not args.keep_all_checkpoints:
            prune_losers(args, st, tag)
    if failed:
        print("\nFAILED runs (strategy, tag, fold):", failed)
    print("\nNext: python sweep.py --phase report")


def phase_report(args):
    df = load_summaries(args.runs_dir)
    if df.empty:
        raise SystemExit(f"No finished runs under {args.runs_dir}")
    os.makedirs("results", exist_ok=True)
    f0 = df[df.fold == 0].sort_values(["model", "strategy", "macro_f1"], ascending=[True, True, False])
    f0.round(4).to_csv("results/sweep_fold0_all_trials.csv", index=False)
    rows = []
    for (m, st), _ in df.groupby(["model", "strategy"]):
        tag = select_tag(df, m, st)
        d0 = df[(df.model == m) & (df.strategy == st) & (df.tag == "default") & (df.fold == 0)]
        sel = df[(df.model == m) & (df.strategy == st) & (df.tag == tag)]
        held = sel[sel.fold.isin(FOLDS_CONFIRM)]
        row = {"model": m, "strategy": st, "selected_tag": tag,
               "default_f0_macro_f1": d0.macro_f1.iloc[0] if len(d0) else float("nan"),
               "selected_f0_macro_f1": sel[sel.fold == 0].macro_f1.mean(),
               "n_heldout_folds": len(held)}
        for k in ["acc", "macro_f1", "bal_acc"]:
            row[f"{k}_mean_f1to4"] = held[k].mean() if len(held) else float("nan")
            row[f"{k}_sd_f1to4"] = held[k].std(ddof=1) if len(held) > 1 else float("nan")
        rows.append(row)
    out = pd.DataFrame(rows)
    out.round(4).to_csv("results/sweep_report.csv", index=False)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    print(out.round(3).to_string(index=False))
    print("\nReport mean +/- SD over held-out folds 1-4 (n_heldout_folds). The fold-0 score of the winner is "
          "optimistic because it was picked on fold 0.")
    print("Saved results/sweep_report.csv and results/sweep_fold0_all_trials.csv")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--phase", choices=["tune", "confirm", "report"], required=True)
    ap.add_argument("--model")
    ap.add_argument("--runs_dir", default="runs")
    ap.add_argument("--include_default", action="store_true",
                    help="confirm: also run the default setting on folds 1-4 when a trial won")
    ap.add_argument("--keep_all_checkpoints", action="store_true",
                    help="confirm: do not delete model.pt of losing tuning trials")
    ap.add_argument("--dry_run", action="store_true")
    args = ap.parse_args()
    if args.phase in ("tune", "confirm") and not args.model:
        ap.error("--model is required for this phase")
    {"tune": phase_tune, "confirm": phase_confirm, "report": phase_report}[args.phase](args)


if __name__ == "__main__":
    main()


In [ ]:
# Check that the model chosen in SETTINGS exists and show its default settings
from common import MODEL_REGISTRY, PRESETS
assert MODEL in MODEL_REGISTRY, f"Unknown MODEL '{MODEL}'. Choose one of {sorted(MODEL_REGISTRY)}"
print(MODEL, "->", MODEL_REGISTRY[MODEL]["timm_name"])
print("partial = unfreeze:", MODEL_REGISTRY[MODEL]["partial_prefixes"], "+ classifier head")
for st in ["frozen", "partial", "full"]:
    print(f"{st:8s}", PRESETS[(MODEL, st)])

## 2. Check the data and make the train/test split
The first cell shows what Kaggle mounted and which folder the scripts will use. If it fails, check that `nail_dataset` is added under *Add Input*, or set `DATA_DIR` explicitly.

In [ ]:
import os
print("/kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "not on Kaggle")
from common import find_data_dir
DATA_DIR = find_data_dir()          # or: DATA_DIR = "/kaggle/input/nail_dataset/dataset_v2_main"
os.environ["NAIL_DATA_DIR"] = DATA_DIR   # train.py, evaluate.py and sweep.py pick this up
print("Using dataset folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
!python split_dataset.py --test_fold 0

## 3. Smoke test and timing (2 epochs, saved to a separate folder)
Checks that this model loads and trains, and shows the seconds per epoch on this GPU. **If this cell fails the notebook stops here on purpose**, so no hours are wasted. Send the error message.

In [ ]:
run(f"python train.py --model {MODEL} --strategy full --epochs 2 --warmup_epochs 1 "
    f"--out_dir runs_timing --save_checkpoint none --overwrite")

## 4. Tune on fold 0
For every strategy: the default run plus 3 trials (frozen: head LR x3, x3 + 60 epochs, x3 + 60 epochs + lighter augmentation; partial and full: backbone LR x0.3, x3, and a stronger-regularisation recipe). Finished runs are skipped, so this cell can be re-run safely.

In [ ]:
!python sweep.py --model {MODEL} --phase tune

### 4b. Extra experiments (optional - edit and run **before** step 6)
Add one line per experiment: `(strategy, tag, "extra train.py arguments")`. Each runs on fold 0 with the same split. If an extra experiment has the best fold-0 macro-F1 for its strategy it is picked up and confirmed on folds 1-4 automatically.

Arguments you can change: `--epochs --lr_head --lr_backbone --weight_decay --warmup_epochs --batch_size --label_smoothing --grad_clip --rrc_scale_min --rotation --jitter --drop_rate --min_lr_ratio --loss {weighted_ce,ce,sampler}`.
To repeat a run with another seed (to measure noise) start the tag with `seed_`, for example `("full", "seed_1", "--seed 1")`: such runs are never selected as winners.

In [ ]:
EXTRA = [
    # ("partial", "lr_x10",  "--lr_backbone 1e-3"),
    # ("full",    "ls0",     "--label_smoothing 0"),
    # ("full",    "sampler", "--loss sampler"),
    # ("full",    "seed_1",  "--seed 1"),
]

import shlex
for strategy, tag, extra_args in EXTRA:
    cmd = (f"python train.py --model {MODEL} --strategy {strategy} --tag {tag} "
           f"--split_csv splits/split_fold0.csv --save_checkpoint full {extra_args}")
    try:
        run(cmd)
    except RuntimeError as e:
        print("FAILED:", e)
print(f"{len(EXTRA)} extra experiment(s) processed")

## 5. Compare all trials on fold 0
`evaluate.py` scores every finished run: accuracy, macro-F1, balanced accuracy, per-class recall, BASE vs external, McNemar tests, confusion matrices and learning curves.

In [ ]:
!python evaluate.py

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
cols = ["run", "acc", "macro_f1", "bal_acc", "acc_BASE", "acc_external", "params_trainable_M", "train_time_min"]
display(pd.read_csv("results/summary.csv")[cols].round(3))
display(pd.read_csv("results/per_class_recall.csv").round(2))
display(Image("results/learning_curves.png"))
for p in sorted(glob.glob("results/confusion_*-default-f0.png")):
    display(Image(p))

## 6. Confirm the winners on folds 1-4
For each strategy the best fold-0 setting (read from its saved `config.json`) is re-run on folds 1-4. Losing trials' checkpoints are deleted to save disk; winners' checkpoints are kept for the XAI work. Add `--include_default` to also run the default setting on folds 1-4 (slower, for a direct tuned-vs-default comparison).

In [ ]:
!python sweep.py --model {MODEL} --phase confirm

## 7. Final report
Mean +/- SD over folds 1-4 for the selected setting of each strategy, plus the fold-0 score of every trial.

In [ ]:
!python sweep.py --phase report

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
display(pd.read_csv("results/sweep_report.csv").round(3))
display(pd.read_csv("results/sweep_fold0_all_trials.csv").round(3))

## 8. Package results (no images, no checkpoints)
Download the zip from the **Output** tab of the saved version. Checkpoints stay in `runs/` of that version.

In [ ]:
import zipfile, os, glob
ZIP = f"nail_sweep_{MODEL}.zip"
KEEP = {"config.json", "summary.json", "epoch_log.csv", "predictions.csv"}
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("results/*"):
        z.write(p)
    for p in glob.glob("runs/**/*", recursive=True):
        if os.path.basename(p) in KEEP:
            z.write(p)
    z.write("splits/class_map.json")
print(ZIP, round(os.path.getsize(ZIP) / 1e6, 2), "MB")